# Recommandation d'animes basée sur le contenu

L'objectif de cette partie est de construire un moteur de recommandation basé sur le contenu qui calcule la similarité entre les animes en fonction de leurs genres (`anime.csv`).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import html
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import kagglehub
from kagglehub import KaggleDatasetAdapter

animes_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "anime.csv"
)
ratings_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "rating.csv"
)

animes = animes_df[['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']].copy()
ratings = ratings_df[['user_id', 'anime_id', 'rating']].copy()

animes['name'] = animes['name'].astype(str).apply(html.unescape)
animes.head()

/home/yzheng1/Bureau/sys_rec/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 915k/915k [00:00<00:00, 1.52MB/s]


100%|██████████| 24.7M/24.7M [00:02<00:00, 11.7MB/s]

Extracting zip of rating.csv...


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama',"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Remplacer les genres manquants par une chaîne vide
animes['genre'] = animes['genre'].fillna('').astype(str)

# Vectorisation TF-IDF sur les genres
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(animes['genre'])
print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)

Dimensions de la matrice TF-IDF : (12294, 654)
